# 📊 **Loan Approval Classification**

Dataset ini memberikan overview mengenai **pengajuan pinjaman**, termasuk atribut finansial dan demografis yang membantu menentukan apakah **pinjaman disetujui atau tidak**. Data ini mencakup fitur yang berkaitan dengan debitur dan jumlah pinjaman, yang dapat digunakan untuk memodelkan probabilitas persetujuan pinjaman.

- **🔢 Jumlah data**: 45,000 record
- **🧩 Total Features**: 14 (mix of Categorical and Continuous types)

### 📝 Feature Descriptions:

| Column                               | Description                                                 | Data Type    |
|--------------------------------------|-------------------------------------------------------------|--------------|
| 👤 **person_age**                     | Usia debitur                                            | Float        |
| 🚻 **person_gender**                  | Jenis kelamin debitur                                          | Categorical  |
| 🎓 **person_education**               | Tingkat pendidikan tertinggi debitur                      | Categorical  |
| 💰 **person_income**                  | Pendapatan tahunan pemohon dalam currency                       | Float        |
| 📅 **person_emp_exp**                 | Lama pengalaman kerja (dalam tahun)                              | Integer      |
| 🏠 **person_home_ownership**          | Status kepemilikan rumah (e.g., rent, own, mortgage)           | Categorical  |
| 🏦 **loan_amnt**                      | Jumlah pinjaman yang diminta                                    | Float        |
| 🎯 **loan_intent**                    | Tujuan pengajuan pinjaman (e.g., personal, education)    | Categorical  |
| 📈 **loan_int_rate**                  | Suku bunga yang dikenakan pada pinjaman                        | Float        |
| 📊 **loan_percent_income**            | Jumlah pinjaman sebagai persentase dari pendapatan tahunan                | Float        |
| 🕰️ **cb_person_cred_hist_length**     | Lama riwayat kredit (dalam tahun)                           | Float        |
| 💳 **credit_score**                   | Skor kredit debitur                                    | Integer      |
| ❗ **previous_loan_defaults_on_file**  | Indikator gagal bayar pinjaman sebelumnya (Yes/No)                | Categorical  |
| ✅ **loan_status**                    | Status hasil pinjaman (1 = approved, 0 = rejected)            | Integer       |



## **Imports and Setup**

In [ ]:
# library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

from sklearn.utils import resample
from sklearn.preprocessing import RobustScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings("ignore")

# 🗺️ **Load and Explore Dataset**

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# API my kaggle
! mkdir ~/.kaggle
! cp '/content/drive/MyDrive/Colab Notebooks/Kaggle API/kaggle.json' ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json
! ls ~/.kaggle

kaggle.json


In [ ]:
# source dataset -> https://www.kaggle.com/datasets/taweilo/loan-approval-classification-data/
! kaggle datasets download taweilo/loan-approval-classification-data/

Dataset URL: https://www.kaggle.com/datasets/taweilo/loan-approval-classification-data/versions/
License(s): apache-2.0
100% 751k/751k [00:00<00:00, 73.5MB/s]



In [ ]:
# unzip
! unzip loan-approval-classification-data.zip -d /content/data/

Archive:  loan-approval-classification-data.zip
  inflating: /content/data/loan_data.csv  


In [ ]:
# load the dataset
df = pd.read_csv('/content/data/loan_data.csv')
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


## **Dataset Information**

In [ ]:
# Basic information about the dataset
print("Dataset Information:")
print(df.info())
print("\nStatistical Summary:")
display(df.describe().T)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_lo

,count,mean,std,min,25%,50%,75%,max
person_age,45000.0,27.764178,6.045108,20.00,24.00,26.00,30.00,144.00
person_income,45000.0,80319.053222,80422.498632,8000.00,47204.00,67048.00,95789.25,7200766.00
person_emp_exp,45000.0,5.410333,6.063532,0.00,1.00,4.00,8.00,125.00
loan_amnt,45000.0,9583.157556,6314.886691,500.00,5000.00,8000.00,12237.25,35000.00
loan_int_rate,45000.0,11.006606,2.978808,5.42,8.59,11.01,12.99,20.00
loan_percent_income,45000.0,0.139725,0.087212,0.00,0.07,0.12,0.19,0.66
cb_person_cred_hist_length,45000.0,5.867489,3.879702,2.00,3.00,4.00,8.00,30.00
credit_score,45000.0,632.608756,50.435865,390.00,601.00,640.00,670.00,850.00
loan_status,45000.0,0.222222,0.415744,0.00,0.00,0.00,0.00,1.00


**Summary**

- **Shape of the dataset**: (45,000, 14) - Dataset berisi 45.000 baris dan 14 kolom.
  
- **Columns and Data Types**:
   - `float64`: person_age, person_income, loan_amnt, loan_int_rate, loan_percent_income, cb_person_cred_hist_length
   - `int64`: person_emp_exp, credit_score, loan_status
   - `object`: person_gender, person_education, person_home_ownership, loan_intent, previous_loan_defaults_on_file

- **Statistical Summary**:
   - `person_age` berada pada rentang 20 hingga 144, yang menunjukkan adanya kemungkinan pencilan (outlier) untuk usia > 100.
   - `person_income` memiliki nilai rata‑rata sekitar 80,319 tetapi juga memiliki nilai maksimum yang sangat tinggi (7,200,766), yang mengindikasikan adanya pencilan pada pendapatan.
   - `loan_amnt` memiliki nilai median 8,000, dengan nilai maksimum 35,000.
   - `credit_score` berada pada rentang 390 hingga 850, yang merupakan rentang umum untuk skor kredit.
   - `loan_status` menunjukkan bahwa sekitar 22% pinjaman disetujui!

## **Duplicate & Missing Values**

In [ ]:
# Check for missing and duplicated values
print(f'Missing values: {df.isna().sum().sum()} records')
print(f'Duplicated values: {df.duplicated().sum()} records')

Missing values: 0 records
Duplicated values: 0 records


Dataset tidak memiliki nilai yang hilang maupun data yang terduplikasi:

- **Missing values**: 0
- **Duplicated values**: 0

## **Unique Value Exploration**

In [ ]:
# Display the number of unique values in each column
print("Unique Values in Each Column:")
print(df.nunique())

Unique Values in Each Column:
person_age                           60
person_gender                         2
person_education                      5
person_income                     33989
person_emp_exp                       63
person_home_ownership                 4
loan_amnt                          4483
loan_intent                           6
loan_int_rate                      1302
loan_percent_income                  64
cb_person_cred_hist_length           29
credit_score                        340
previous_loan_defaults_on_file        2
loan_status                           2
dtype: int64


In [ ]:
# Separate numerical and categorical columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
non_numerical_columns = df.select_dtypes(include=['object']).columns.tolist()

# Display the lists of numerical and categorical columns
print("Numerical Columns:", numerical_columns)
print("Categorical Columns:", non_numerical_columns)

Numerical Columns: ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'loan_status']
Categorical Columns: ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']


In [ ]:
# Display unique values for each categorical column
print(f"Categorical Values: ")
for col in non_numerical_columns:
    print(f"\nColumn >> {col}")
    print(f"Unique Values: {df[col].unique()}")

Categorical Values: 

Column >> person_gender
Unique Values: ['female' 'male']

Column >> person_education
Unique Values: ['Master' 'High School' 'Bachelor' 'Associate' 'Doctorate']

Column >> person_home_ownership
Unique Values: ['RENT' 'OWN' 'MORTGAGE' 'OTHER']

Column >> loan_intent
Unique Values: ['PERSONAL' 'EDUCATION' 'MEDICAL' 'VENTURE' 'HOMEIMPROVEMENT'
 'DEBTCONSOLIDATION']

Column >> previous_loan_defaults_on_file
Unique Values: ['No' 'Yes']


**Summary terkait Unique Values dan Data Types pada dataset ini:**

**Unique Values Setiap Features**

- **person_age**: 60 unique values
- **person_gender**: 2 unique values (`female`, `male`)
- **person_education**: 5 unique values (`Master`, `High School`, `Bachelor`, `Associate`, `Doctorate`)
- **person_income**: 33,989 unique values
- **person_emp_exp**: 63 unique values
- **person_home_ownership**: 4 unique values (`RENT`, `OWN`, `MORTGAGE`, `OTHER`)
- **loan_amnt**: 4,483 unique values
- **loan_intent**: 6 unique values (`PERSONAL`, `EDUCATION`, `MEDICAL`, `VENTURE`, `HOMEIMPROVEMENT`, `DEBTCONSOLIDATION`)
- **loan_int_rate**: 1,302 unique values
- **loan_percent_income**: 64 unique values
- **cb_person_cred_hist_length**: 29 unique values
- **credit_score**: 340 unique values
- **previous_loan_defaults_on_file**: 2 unique values (`No`, `Yes`)
- **loan_status**: 2 unique values (target variable)

**Column Types**

- **Numerical Columns**: `person_age`, `person_income`, `person_emp_exp`, `loan_amnt`, `loan_int_rate`, `loan_percent_income`, `cb_person_cred_hist_length`, `credit_sc
  ore`, `loan_status`
- **Categorical Columns**: `person_gender`, `person_education`, `person_home_ownership`, `loan_intent`, `previous_loath any specific steps!

# ✨ **Exploratory Data Analysis (EDA)**

In [ ]:
# Count the occurrences of each loan status
loan_status_counts = df['loan_status'].value_counts().sort_index()

# Create label mapping
status_labels = {
    0: 'Rejected (0)',
    1: 'Approved (1)'
}

labels = [status_labels.get(i, str(i)) for i in loan_status_counts.index]
values = loan_status_counts.values

# Colors for each bar and pie slice
colors = px.colors.qualitative.Plotly[:len(labels)]

# Create subplots: bar chart and pie chart
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        'Distribution of Loan Approval Status',
        'Percentage Distribution of Loan Approval Status'
    ),
    specs=[[{"type": "xy"}, {"type": "domain"}]],
    horizontal_spacing=0.18
)

# Bar plot
fig.add_trace(
    go.Bar(
        x=labels,
        y=values,
        text=values,
        textposition='outside',
        marker_color=colors,
        name='Loan Status Count',
        cliponaxis=False
    ),
    row=1,
    col=1
)

# Pie chart
fig.add_trace(
    go.Pie(
        labels=labels,
        values=values,
        textinfo='label+percent',
        textfont=dict(color='white'),
        marker=dict(colors=colors),
        name='Loan Status Percentage'
    ),
    row=1,
    col=2
)

# Layout settings
fig.update_layout(
    title=dict(
        text='Loan Approval Status Visualization',
        x=0.5,
        xanchor='center',
        y=0.97,
        yanchor='top'
    ),
    height=600,
    margin=dict(
        t=120,
        b=60,
        l=70,
        r=70
    ),
    legend=dict(
        orientation='h',
        xanchor='center',
        x=0.5
    ),
    showlegend=False
)

# Adjust subplot title position
for annotation in fig.layout.annotations:
    annotation.update(
        y=1.08,
        yanchor='bottom'
    )

# Axis labels
fig.update_xaxes(
    title_text='Loan Status',
    row=1,
    col=1
)

fig.update_yaxes(
    title_text='Count',
    row=1,
    col=1
)

# Add extra y-axis space so bar labels are not too close to the subplot title
fig.update_yaxes(
    range=[0, max(values) * 1.25],
    row=1,
    col=1
)

fig.show()

***Insight dari visualisasi loan status distribution:***

* Anotasi jumlah di atas setiap batang membantu menunjukkan dengan jelas berapa banyak data di setiap kategori, sehingga terlihat jika ada ketidakseimbangan dalam dataset.

* Visualisasi ini menunjukkan jumlah pinjaman yang ditolak lebih besar daripada yang disetujui, sehingga mengindikasikan adanya ketidakseimbangan kelas yang perlu diperhatikan saat membangun model prediktif.

* Diagram lingkaran menampilkan persentase status persetujuan pinjaman dan memberikan gambaran ringkas mengenai perbandingan antara pinjaman yang disetujui dan yang ditolak.

In [ ]:
# Function to perform univariate analysis for numeric columns using Plotly
def univariate_analysis_plotly(data, columns, bins=30):

    # Create subplot titles
    subplot_titles = [
        f'{col.replace("_", " ").title()} Distribution with KDE'
        for col in columns
    ]

    # Create subplots
    fig = make_subplots(
        rows=4,
        cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.1,
        horizontal_spacing=0.12
    )

    for i, column in enumerate(columns):
        row = (i // 2) + 1
        col = (i % 2) + 1

        # Remove missing values
        series = data[column].dropna()

        # Create histogram manually
        counts, bin_edges = np.histogram(series, bins=bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        # Histogram
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=counts,
                width=bin_width,
                opacity=0.75,
                name='Histogram',
                showlegend=True if i == 0 else False
            ),
            row=row,
            col=col
        )

        # KDE line, scaled to frequency
        kde = gaussian_kde(series)
        x_range = np.linspace(series.min(), series.max(), 300)
        kde_values = kde(x_range) * len(series) * bin_width

        fig.add_trace(
            go.Scatter(
                x=x_range,
                y=kde_values,
                mode='lines',
                name='KDE',
                showlegend=True if i == 0 else False
            ),
            row=row,
            col=col
        )

        # Axis labels
        fig.update_xaxes(
            title_text=column.replace('_', ' ').title(),
            title_standoff=12,
            row=row,
            col=col
        )

        fig.update_yaxes(
            title_text='Frequency',
            title_standoff=12,
            row=row,
            col=col
        )

    # Layout settings
    fig.update_layout(
        title=dict(
            text='Univariate Analysis of Numeric Features',
            x=0.5,
            xanchor='center',
            y=0.98,
            yanchor='top'
        ),
        height=1750,
        margin=dict(
            t=140,
            b=80,
            l=80,
            r=60
        ),
        bargap=0.05,
        showlegend=True,
        legend=dict(
            orientation='h',
            x=0.5,
            xanchor='center',
            y=1.04,
            yanchor='bottom'
        )
    )

    # Adjust subplot title position
    for annotation in fig.layout.annotations:
        annotation.update(
            y=annotation.y + 0.025,
            font=dict(size=13)
        )

    fig.show()


columns_to_analyze = [
    'person_age',
    'person_income',
    'person_emp_exp',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length',
    'credit_score'
]

univariate_analysis_plotly(df, columns_to_analyze)

***Beberapa insights dari univariate analysis:***

1. **Person Age**:
   - Distribusi usia sedikit miring ke kanan (slightly right-skewed), dengan sebagian besar individu dalam rentang usia 20 hingga 40 tahun.
   - Visualisasi KDE (Kernel Density Estimate) membantu menghaluskan distribusi sehingga rentang usia yang umum lebih mudah terlihat.

2. **Person Income**:
   - Distribusi pendapatan sangat miring ke kanan (highly right-skewed), dengan banyak pendapatan terkonsentrasi pada nilai yang lebih rendah.
   - Terdapat beberapa nilai pendapatan yang sangat tinggi, yang menunjukkan adanya outlier yang dapat memengaruhi kinerja model jika tidak ditangani.

3. **Person Employment Experience**:
   - Mayoritas individu memiliki pengalaman kerja kurang dari 10 tahun, dengan frekuensi yang menurun tajam seiring bertambahnya tahun pengalaman.
   - Beberapa nilai sangat tinggi kemungkinan merupakan outlier yang dapat mengganggu analisis jika tidak ditangani.

4. **Loan Amount**:
   - Jumlah pinjaman terkonsentrasi pada nilai yang lebih rendah, yang menunjukkan bahwa sebagian besar debitur mengajukan pinjaman dengan nilai kecil.
   - Distribusi menurun secara bertahap, dengan sedikit pemohon yang meminta jumlah pinjaman besar.

5. **Loan Interest Rate**:
   - Suku bunga sebagian besar berkumpul di sekitar 10% hingga 15%, yang sesuai dengan kisaran umum suku bunga pinjaman.
   - Terdapat kepadatan yang cukup besar antara 5% hingga 10%, yang mungkin menunjukkan debitur dengan profil risiko lebih rendah (low-risk profiles).

6. **Loan Percent Income**:
   - Distribusi ini menunjukkan bahwa sebagian besar jumlah pinjaman merupakan persentase kecil dari pendapatan debitur, sering kali kurang dari 20%.
   - Beberapa kasus memiliki persentase yang lebih tinggi, yang mengindikasikan risiko lebih besar atau debitur dengan pendapatan lebih rendah relatif terhadap jumlah pinjamannya.

7. **Credit History Length**:
   - Tenor riwayat kredit memuncak di sekitar 3 hingga 5 tahun, dengan lebih sedikit individu yang memiliki riwayat kredit lebih dari 10 tahun.
   - Pola ini dapat mencerminkan demografi yang lebih muda atau individu yang relatif baru masuk ke sistem kredit.

8. **Credit Score**:
   - Skor kredit terdistribusi secara normal di kisaran menengah (600–700).
   - Distribusi menurun mendekati 850, yang menunjukkan hanya sedikit individu dengan skor kredit sangat tinggi.

In [ ]:
# Function to perform univariate analysis for numeric columns
def univariate_analysis_plotly(data, column, title):
    fig = px.box(
        data_frame=data,
        x=column,
        title=f'{title} Boxplot',
        template='plotly_white',
        width=800,
        height=250
    )
    fig.show()

    print(f'\nSummary Statistics for {title}:\n', data[column].describe())
    print('\n')


# List of columns to analyze
columns_to_analyze = [
    'person_age',
    'person_income',
    'person_emp_exp',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length',
    'credit_score'
]

for column in columns_to_analyze:
    univariate_analysis_plotly(df, column, column.replace('_', ' '))

***Insights berdasarkan boxplots and summary statistics:***

1. **Person Age**:
   - Rentang usia antara 20 hingga 144 tahun, dengan median 26 tahun. Nilai maksimum yang sangat tinggi menunjukkan adanya beberapa outlier.
   - Rentang antarkuartil (IQR) cukup sempit, dengan sebagian besar nilai berada antara 24 dan 30 tahun.

2. **Person Income**:
   - Distribusi pendapatan memiliki rentang yang sangat lebar, dari 8,000 hingga lebih dari 7.2 juta, dengan median sekitar 67,048.
   - Nilai maksimum yang sangat tinggi mengindikasikan outlier ekstrem yang berpotensi menggeser hasil analisis dan model.

3. **Person Employment Experience**:
   - Sebagian besar nilai berada di bawah 10 tahun, dengan median pengalaman kerja 4 tahun.
   - Nilai maksimum 125 tahun tidak wajar dan menunjukkan adanya outlier atau anomali data.

4. **Loan Amount**:
   - Jumlah pinjaman berkisar antara 500 hingga 35,000, dengan median 8,000.
   - Distribusi menunjukkan sebaran yang cukup luas, dengan beberapa pemohon mengajukan pinjaman mendekati batas atas.

5. **Loan Interest Rate**:
   - Suku bunga berkisar antara 5.42% hingga 20%, dengan median 11.01%.
   - Sebagian besar suku bunga berada dalam IQR 8.59% hingga 12.99%, yang masih sesuai dengan kisaran umum suku bunga pinjaman.

6. **Loan Percent Income**:
   - Metrik ini berkisar antara 0 hingga 0.66, dengan median 0.12, yang menunjukkan bahwa sebagian besar pinjaman kurang dari 20% dari pendapatan peminjam.
   - Nilai yang mendekati 0.66 mengindikasikan pinjaman yang menjadi beban finansial lebih besar bagi sebagian debitur.

7. **Credit History Length**:
   - Credit history spans from 2 to 30 years, with a median of 4 years.
   - Most applicants have shorter credit histories, likely reflecting a younger demographic.
   - Riwayat kredit berada dalam rentang 2 hingga 30 tahun, dengan median 4 tahun.
   - Sebagian besar debitur memiliki riwayat kredit yang relatif pendek, kemungkinan mencerminkan demografi yang lebih muda.

8. **Credit Score**:
   - Skor kredit berkisar antara 390 hingga 850, dengan median 640.
   - Distribusinya tampak cukup simetris di sekitar mean 632, dengan sebagian besar nilai berada dalam rentang tengah skor kredit.

In [ ]:
def plot_categorical_distribution_plotly(column_name, data=df):
    counts = data[column_name].value_counts().reset_index()
    counts.columns = [column_name, 'count']

    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.55, 0.45],
        specs=[[{"type": "bar"}, {"type": "pie"}]],
        subplot_titles=(
            f"Distribution of {column_name}",
            f"Percentage Distribution of {column_name}"
        )
    )

    fig.add_trace(
        go.Bar(
            x=counts['count'],
            y=counts[column_name],
            orientation='h',
            marker_color='teal',
            text=counts['count'],
            textposition='outside'
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Pie(
            labels=counts[column_name],
            values=counts['count'],
            hole=0,
            textinfo='percent',
            pull=[0.05] * len(counts)
        ),
        row=1, col=2
    )

    fig.update_layout(
        template='plotly_white',
        showlegend=False,
        height=450,
        title_x=0.5,
        margin=dict(l=60, r=40, t=80, b=60)
    )

    fig.show()

plot_categorical_distribution_plotly('person_gender')
plot_categorical_distribution_plotly('person_education')
plot_categorical_distribution_plotly('person_home_ownership')
plot_categorical_distribution_plotly('loan_intent')
plot_categorical_distribution_plotly('previous_loan_defaults_on_file')

***Insights dari distribusi fitur categorical:***

1. **Person Gender**:
   - Dataset ini relatif seimbang dari sisi jenis kelamin, meskipun mungkin terdapat sedikit kecenderungan ke salah satu gender, bergantung pada jumlah masing‑masing.
   - Diagram lingkaran memberikan ringkasan visual mengenai proporsi setiap gender dalam dataset.

2. **Person Education**:
   - Sebagian besar debitur memiliki pendidikan SMA, sarjana, atau magister, sementara pemegang gelar diploma (associate) atau doktor jumlahnya lebih sedikit.
   - Distribusi tingkat pendidikan ini dapat memengaruhi pola persetujuan pinjaman, karena tingkat pendidikan sering berkorelasi dengan pendapatan dan kelayakan kredit.

3. **Person Home Ownership**:
   - Mayoritas debitur berada pada status sewa atau rumah milik sendiri, dengan jumlah lebih kecil yang memiliki hipotek atau diklasifikasikan sebagai “other”.
   - Debitur yang memiliki status kepemilikan rumah yang berbeda-beda, kemungkinan memiliki stabilitas finansial yang berbeda pula, sehingga berdampak pada risiko kredit.

4. **Loan Intent**:
   - Tujuan pengajuan pinjaman cukup beragam, dengan alasan umum seperti kebutuhan pribadi, konsolidasi utang, biaya medis, dan pendidikan.
   - Distribusi ini menunjukkan alasan utama pengajuan pinjaman, yang dapat memengaruhi kriteria persetujuan tergantung tingkat risiko masing‑masing tujuan.

5. **Previous Loan Defaults on File**:
   - Sebagian besar debitur tidak memiliki catatan gagal bayar pinjaman sebelumnya, meskipun ada porsi yang signifikan dengan riwayat gagal bayar.
   - Fitur ini dapat sangat memengaruhi keputusan pemberian pinjaman, karena gagal bayar di masa lalu menunjukkan risiko yang lebih tinggi.


In [ ]:
# categorical features
def add_bar_subplot(fig, df, x_col, row, col, title):
    counts = (
        df.groupby([x_col, 'loan_status'])
          .size()
          .reset_index(name='count')
    )

    status_map = {0: 'Rejected', 1: 'Approved'}
    counts['loan_status_label'] = counts['loan_status'].map(status_map)

    max_count = counts['count'].max()
    upper = max_count * 1.8

    fig.add_trace(
        go.Bar(
            x=counts[x_col],
            y=counts['count'],
            marker_color=None,
            text=counts['count'],
            textposition='outside',
            customdata=counts[['loan_status_label']].values,
            hovertemplate=(
                x_col.replace('_', ' ').title() + ': %{x}<br>'
                'Loan Status: %{customdata[0]}<br>'
                'Count: %{y}<extra></extra>'
            ),
            showlegend=False
        ),
        row=row, col=col
    )

    fig.update_xaxes(
        title_text=x_col.replace('_', ' ').title(),
        row=row, col=col
    )
    fig.update_yaxes(
        title_text='Count',
        range=[0, upper],
        row=row, col=col
    )


fig = make_subplots(
    rows=2, cols=3,
    specs=[
        [{"type": "bar"}, {"type": "bar"}, {"type": "bar"}],
        [{"type": "bar", "colspan": 2}, None, {"type": "bar"}]
    ],
    subplot_titles=[
        "Loan Status by Gender",
        "Loan Status by Education Level",
        "Loan Status by Home Ownership",
        "Loan Status by Loan Intent",
        "Loan Status by Previous Loan Defaults"
    ],
    vertical_spacing=0.2
)

add_bar_subplot(fig, df, 'person_gender', row=1, col=1, title='Loan Status by Gender')
add_bar_subplot(fig, df, 'person_education', row=1, col=2, title='Loan Status by Education Level')
add_bar_subplot(fig, df, 'person_home_ownership', row=1, col=3, title='Loan Status by Home Ownership')
add_bar_subplot(fig, df, 'loan_intent', row=2, col=1, title='Loan Status by Loan Intent')
add_bar_subplot(fig, df, 'previous_loan_defaults_on_file', row=2, col=3, title='Loan Status by Previous Loan Defaults')

fig.update_layout(
    height=800,
    template='plotly_white',
    barmode='group',
    title=dict(
        text="Loan Status by Categorical Features",
        x=0.5,
        xanchor='center',
        y=0.97,
        yanchor='top'
    ),
    margin=dict(l=80, r=40, t=110, b=80),
    legend_title_text='Loan Status'
)

fig.show()

***Insights berdasarkan relasi antara **loan status** dengan beberapa fitur ketegorikal lainya:***

1. **Loan Status by Gender**:
   - Persetujuan dan penolakan pinjaman relatif seimbang di antara gender, meskipun ada variasi kecil.
   - Keseimbangan ini mengindikasikan bahwa gender kemungkinan bukan faktor penentu utama dalam hasil persetujuan pinjaman.

2. **Loan Status by Education Level**:
   - Debitur dengan tingkat pendidikan lebih tinggi (seperti sarjana) cenderung memiliki jumlah persetujuan pinjaman yang lebih besar dibandingkan debitur berpendidikan lebih rendah.
   - Tingkat pendidikan berpotensi menjadi prediktor persetujuan pinjaman karena biasanya berkorelasi dengan pendapatan dan kelayakan kredit yang lebih baik.

3. **Loan Status by Home Ownership**:
   - Debitur dengan status sewa terlihat memiliki tingkat penolakan pinjaman yang lebih tinggi dibandingkan mereka yang memiliki hipotek atau rumah sendiri.
   - Pola ini menunjukkan bahwa status kepemilikan rumah dapat dipandang sebagai faktor risiko, karena penyewa mungkin dianggap memiliki stabilitas finansial yang lebih rendah dibanding pemilik rumah.

4. **Loan Status by Loan Intent**:
   - Beberapa tujuan pinjaman seperti debt consolidation dan pinjaman personal menunjukkan jumlah penolakan yang lebih tinggi daripada persetujuan.
   - Sebaliknya, tujuan pinjaman untuk venture dan education terlihat lebih seimbang antara persetujuan dan penolakan, kemungkinan karena dianggap memiliki potensi peningkatan pendapatan atau perbaikan kapasitas finansial.

5. **Loan Status by Previous Loan Defaults**:
   - Debitur dengan riwayat gagal bayar pinjaman sebelumnya memiliki tingkat penolakan yang jauh lebih tinggi dibandingkan debitur tanpa riwayat gagal bayar.
   - Fitur ini sangat berpengaruh terhadap status pinjaman, karena gagal bayar di masa lalu menjadi sinyal risiko kredit yang kuat.

In [ ]:
numerical_columns = [
    'person_age', 'person_income', 'person_emp_exp',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length', 'credit_score'
]

# Buat 4x2 subplot
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=[f"{col} vs Loan Status" for col in numerical_columns]
)

# Loop setiap kolom numerik
for i, col in enumerate(numerical_columns):
    row = i // 2 + 1
    col_pos = i % 2 + 1

    # Histogram untuk loan_status = 0 (Rejected)
    fig.add_trace(
        go.Histogram(
            x=df[df['loan_status'] == 0][col],
            name='Rejected',
            opacity=0.5,
            marker_color='tomato',
            showlegend=(i == 0)
        ),
        row=row, col=col_pos
    )

    # Histogram untuk loan_status = 1 (Approved)
    fig.add_trace(
        go.Histogram(
            x=df[df['loan_status'] == 1][col],
            name='Approved',
            opacity=0.5,
            marker_color='steelblue',
            showlegend=(i == 0)
        ),
        row=row, col=col_pos
    )

    fig.update_xaxes(title_text=col, row=row, col=col_pos)
    fig.update_yaxes(title_text='Density', row=row, col=col_pos)

# histogram saling menumpuk sebagai density-like
fig.update_layout(
    barmode='overlay',
    template='plotly_white',
    height=1500,
    title=dict(
        text='Numerical Features vs Loan Status (Approx. Density Plots)',
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=70, r=40, t=90, b=60)
)

fig.show()

In [ ]:
numerical_columns = [
    'person_age', 'person_income', 'person_emp_exp',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length', 'credit_score'
]

# Buat 8 subplot vertikal
fig = make_subplots(
    rows=len(numerical_columns),
    cols=1,
    shared_xaxes=False,
    subplot_titles=[f"{col} vs Loan Status" for col in numerical_columns]
)

status_map = {0: 'Rejected', 1: 'Approved'}
colors = {0: 'tomato', 1: 'steelblue'}

for i, feature in enumerate(numerical_columns, start=1):
    for status in [0, 1]:
        fig.add_trace(
            go.Box(
                x=df[df['loan_status'] == status]['loan_status']
                  .map(status_map),
                y=df[df['loan_status'] == status][feature],
                name=status_map[status],
                marker_color=colors[status],
                boxmean='sd',
                showlegend=False
            ),
            row=i, col=1
        )

    fig.update_yaxes(title_text=feature, row=i, col=1)
    fig.update_xaxes(title_text='Loan Status', row=i, col=1)

fig.update_layout(
    height=2000,
    template='plotly_white',
    boxmode='group',
    title=dict(
        text='Boxplots of Numerical Features by Loan Status',
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=80, r=40, t=90, b=60),
    legend_title_text='Loan Status'
)

fig.show()

***Insights dari boxplot fitur numerik berdasarkan loan status:***

1. **Person Age**:
   - Pinjaman yang disetujui cenderung memiliki median usia sedikit lebih muda, meskipun perbedaannya tidak terlalu besar.
   - Rentang distribusi usia pada pinjaman yang ditolak lebih lebar, dengan outlier di usia yang sangat tinggi, yang dapat mengindikasikan usia lanjut sebagai faktor risiko kecil.

2. **Person Income**:
   - Pinjaman yang disetujui umumnya diajukan oleh debitur dengan pendapatan lebih tinggi.
   - Median pendapatan untuk pinjaman yang disetujui terlihat jauh lebih besar, dengan banyak outlier berpendapatan tinggi, sehingga memperkuat bahwa pendapatan berkontribusi positif terhadap persetujuan.

3. **Person Employment Experience**:
   - Pengalaman kerja yang lebih lama menunjukkan korelasi ringan dengan persetujuan pinjaman, tercermin pada median pengalaman yang lebih tinggi di kelompok yang disetujui.
   - Namun, kedua kelompok (disetujui dan ditolak) memiliki rentang pengalaman yang luas, sehingga kemungkinan ada faktor lain yang lebih dominan.

4. **Loan Amount**:
   - Besaran pinjaman relatif mirip antara pinjaman yang disetujui dan ditolak, meskipun median di kelompok yang ditolak sedikit lebih tinggi.
   - Hal ini dapat mengindikasikan bahwa pinjaman dengan nilai lebih besar sedikit lebih berisiko untuk ditolak, tetapi perbedaannya tidak terlalu signifikan.

5. **Loan Interest Rate**:
   - Pinjaman yang disetujui cenderung memiliki suku bunga sedikit lebih rendah dibandingkan pinjaman yang ditolak.
   - Pola ini sejalan dengan persepsi risiko yang lebih tinggi pada debitur dengan suku bunga lebih tinggi, yang sering terkait dengan profil kredit yang lebih lemah.

6. **Loan Percent Income**:
   - Debitur dengan pinjaman yang disetujui umumnya memiliki rasio pinjaman terhadap pendapatan yang lebih rendah, sehingga beban cicilan terhadap pendapatan lebih ringan.
   - Rasio yang tinggi pada pinjaman yang ditolak menunjukkan bahwa pemberi pinjaman lebih berhati‑hati ketika jumlah pinjaman membentuk porsi besar dari pendapatan debitur.

7. **Credit History Length**:
   - Riwayat kredit yang lebih panjang lebih sering muncul pada pinjaman yang disetujui, menandakan bahwa debitur dengan rekam jejak kredit lebih lama memiliki peluang persetujuan yang lebih tinggi.
   - Pola ini mencerminkan preferensi lembaga keuangan terhadap peminjam yang sudah terbukti dalam mengelola kredit.

8. **Credit Score**:
   - Pinjaman yang disetujui konsisten terkait dengan skor kredit yang lebih tinggi.
   - Perbedaan yang cukup jelas antara skor pada pinjaman disetujui dan ditolak menegaskan bahwa skor kredit adalah prediktor kuat untuk hasil persetujuan, dengan skor tinggi mencerminkan risiko yang lebih rendah.

In [ ]:
# check infinite values
np.isinf(df[numerical_columns]).sum()

- Tidak ditemukan nilai tak hingga (infinite) pada kolom numerik mana pun. Setiap kolom di dalam `numerical_columns` memiliki 0 kemunculan nilai tak hingga.

In [ ]:
# Copy data
plot_df = df.copy()

# Label mapping for loan status
status_labels = {
    0: '0 = Rejected',
    1: '1 = Approved',
    '0': '0 = Rejected',
    '1': '1 = Approved'
}

plot_df['loan_status_label'] = plot_df['loan_status'].map(status_labels).fillna(
    plot_df['loan_status'].astype(str)
)

# Count data by gender, education, and loan status
count_df = (
    plot_df
    .groupby(['person_gender', 'person_education', 'loan_status_label'])
    .size()
    .reset_index(name='count')
)

# Get category orders
genders = plot_df['person_gender'].dropna().unique()
education_order = plot_df['person_education'].dropna().unique()
loan_status_order = ['0 = Rejected', '1 = Approved']

# Colors
colors = px.colors.qualitative.Plotly[:len(loan_status_order)]
color_map = dict(zip(loan_status_order, colors))

# Create subplots by gender
fig = make_subplots(
    rows=1,
    cols=len(genders),
    subplot_titles=[f'Gender: {gender}' for gender in genders],
    horizontal_spacing=0.12
)

# Add grouped bar charts
for col_idx, gender in enumerate(genders, start=1):
    gender_df = count_df[count_df['person_gender'] == gender]

    for status in loan_status_order:
        status_df = gender_df[gender_df['loan_status_label'] == status]

        # Make sure all education categories appear even if count is 0
        status_df = (
            pd.DataFrame({'person_education': education_order})
            .merge(status_df, on='person_education', how='left')
        )
        status_df['count'] = status_df['count'].fillna(0).astype(int)

        fig.add_trace(
            go.Bar(
                x=status_df['person_education'],
                y=status_df['count'],
                text=status_df['count'],
                textposition='outside',
                name=status,
                marker_color=color_map[status],
                showlegend=True if col_idx == 1 else False,
                cliponaxis=False
            ),
            row=1,
            col=col_idx
        )

    fig.update_xaxes(
        title_text='Education Level',
        title_standoff=15,
        categoryorder='array',
        categoryarray=education_order,
        row=1,
        col=col_idx
    )

    fig.update_yaxes(
        title_text='Count' if col_idx == 1 else '',
        title_standoff=15,
        row=1,
        col=col_idx
    )

# Add extra y-axis space for labels
max_count = count_df['count'].max()
for col_idx in range(1, len(genders) + 1):
    fig.update_yaxes(
        range=[0, max_count * 1.25],
        row=1,
        col=col_idx
    )

# Layout settings
fig.update_layout(
    title=dict(
        text='Loan Approval Status by Education Level and Gender',
        x=0.5,
        xanchor='center',
        y=0.98,
        yanchor='top'
    ),
    height=600,
    margin=dict(
        t=150,
        b=90,
        l=80,
        r=70
    ),
    barmode='group',
    bargap=0.25,
    bargroupgap=0.08,
    showlegend=True,
    legend=dict(
        title='Loan Status',
        orientation='h',
        x=0.5,
        xanchor='center',
        y=1.08,
        yanchor='bottom'
    )
)

# Adjust subplot title spacing
for annotation in fig.layout.annotations:
    annotation.update(
        y=1.02,
        yanchor='bottom',
        font=dict(size=14)
    )

fig.show()

***Insights untuk **education level vs. loan status by gender**:***

1. **Gender Comparison**:
   - Baik debitur laki‑laki maupun perempuan menunjukkan pola yang mirip dalam tingkat persetujuan dan penolakan pinjaman di setiap jenjang pendidikan, meskipun terdapat sedikit variasi.

2. **Education Level and Loan Status**:
   - Pada kedua gender, debitur dengan tingkat pendidikan lebih tinggi (misalnya sarjana, magister) cenderung memiliki jumlah persetujuan pinjaman yang lebih besar.
   - Debitur yang hanya memiliki pendidikan setara SMA tampak mengalami lebih banyak penolakan dibanding persetujuan, yang mengindikasikan bahwa tingkat pendidikan dapat memengaruhi hasil pinjaman, kemungkinan karena kaitannya dengan stabilitas pendapatan.

3. **Approval Pattern by Gender**:
   - Pada laki‑laki maupun perempuan, terlihat tren bahwa debitur dengan pendidikan lebih tinggi (seperti pemegang gelar sarjana dan magister) lebih berpeluang mendapat persetujuan pinjaman.
   - Hal ini menyiratkan bahwa tingkat pendidikan menjadi indikator positif bagi persetujuan pinjaman, yang mencerminkan profil finansial yang dinilai lebih stabil.

4. **Rejected Applications**:
   - Tingkat penolakan lebih tinggi terjadi pada debitur dengan tingkat pendidikan yang lebih rendah, yang dapat menunjukkan persepsi risiko yang lebih tinggi dari sisi pemberi pinjaman.

Secara keseluruhan, pola ini menunjukkan bahwa tingkat pendidikan merupakan faktor yang berpengaruh dalam keputusan persetujuan pinjaman dan bekerja dengan cara yang relatif serupa pada kedua gender.

In [ ]:
# Copy data
plot_df = df.copy()

# Label mapping for loan status
status_labels = {
    0: '0 = Rejected',
    1: '1 = Approved',
    '0': '0 = Rejected',
    '1': '1 = Approved'
}

plot_df['loan_status_label'] = plot_df['loan_status'].map(status_labels).fillna(
    plot_df['loan_status'].astype(str)
)

# Count data by gender, home ownership, and loan status
count_df = (
    plot_df
    .groupby(['person_gender', 'person_home_ownership', 'loan_status_label'])
    .size()
    .reset_index(name='count')
)

# Category orders
genders = plot_df['person_gender'].dropna().unique()
home_order = plot_df['person_home_ownership'].dropna().unique()
loan_status_order = ['0 = Rejected', '1 = Approved']

# Colors
colors = px.colors.qualitative.Plotly[:len(loan_status_order)]
color_map = dict(zip(loan_status_order, colors))

# Create subplots by gender
fig = make_subplots(
    rows=1,
    cols=len(genders),
    subplot_titles=[f'Gender: {gender}' for gender in genders],
    horizontal_spacing=0.12
)

# Add grouped bar charts
for col_idx, gender in enumerate(genders, start=1):
    gender_df = count_df[count_df['person_gender'] == gender]

    for status in loan_status_order:
        status_df = gender_df[gender_df['loan_status_label'] == status]

        # Ensure all home ownership categories appear
        status_df = (
            pd.DataFrame({'person_home_ownership': home_order})
            .merge(status_df, on='person_home_ownership', how='left')
        )

        status_df['count'] = status_df['count'].fillna(0).astype(int)

        fig.add_trace(
            go.Bar(
                x=status_df['person_home_ownership'],
                y=status_df['count'],
                text=status_df['count'],
                textposition='outside',
                name=status,
                marker_color=color_map[status],
                showlegend=True if col_idx == 1 else False,
                cliponaxis=False
            ),
            row=1,
            col=col_idx
        )

    fig.update_xaxes(
        title_text='Home Ownership',
        title_standoff=15,
        categoryorder='array',
        categoryarray=home_order,
        tickangle=0,
        row=1,
        col=col_idx
    )

    fig.update_yaxes(
        title_text='Count' if col_idx == 1 else '',
        title_standoff=15,
        row=1,
        col=col_idx
    )

# Add extra y-axis space for count labels
max_count = count_df['count'].max()

for col_idx in range(1, len(genders) + 1):
    fig.update_yaxes(
        range=[0, max_count * 1.25],
        row=1,
        col=col_idx
    )

# Layout settings
fig.update_layout(
    title=dict(
        text='Loan Approval Status by Home Ownership and Gender',
        x=0.5,
        xanchor='center',
        y=0.98,
        yanchor='top'
    ),
    height=600,
    # width=1200,
    margin=dict(
        t=150,
        b=90,
        l=80,
        r=70
    ),
    barmode='group',
    bargap=0.25,
    bargroupgap=0.08,
    showlegend=True,
    legend=dict(
        title='Loan Status',
        orientation='h',
        x=0.5,
        xanchor='center',
        y=1.08,
        yanchor='bottom'
    )
)

# Adjust subplot title spacing
for annotation in fig.layout.annotations:
    annotation.update(
        y=1.02,
        yanchor='bottom',
        font=dict(size=14)
    )

fig.show()

***Insights untuk **home ownership vs. loan status by gender**:***

1. **Home Ownership and Loan Status**:
   - Baik debitur laki‑laki maupun perempuan yang memiliki rumah sendiri atau sedang KPR (mortgage) menunjukkan jumlah persetujuan pinjaman yang lebih tinggi dibandingkan mereka yang menyewa atau masuk kategori “other”.
   - Status menyewa dikaitkan dengan tingkat penolakan yang lebih tinggi, yang kemungkinan mencerminkan persepsi risiko kredit yang lebih besar terhadap penyewa.

2. **Gender-Specific Patterns**:
   - Meskipun polanya serupa di kedua gender, debitur laki‑laki yang memiliki rumah atau KPR tampak memiliki jumlah persetujuan pinjaman sedikit lebih tinggi dibanding perempuan dengan status kepemilikan rumah yang sama.
   - Pada kedua gender, debitur dalam kategori “other” relatif jarang mendapatkan persetujuan, yang mengindikasikan bahwa stabilitas kepemilikan rumah (milik sendiri atau KPR) menjadi faktor positif bagi persetujuan pinjaman.

3. **Rejected Applications**:
   - Di kalangan penyewa, tingkat penolakan pinjaman terlihat cukup tinggi pada kedua gender, yang dapat mengisyaratkan bahwa penyewa dianggap memiliki tingkat keamanan finansial atau kelayakan kredit yang lebih rendah dibanding pemilik rumah.

4. **Homeownership as a Stability Indicator**:
   - Data menunjukkan bahwa debitur yang memiliki rumah atau KPR dinilai lebih menguntungkan, kemungkinan karena mereka dipandang lebih stabil secara finansial, sehingga peluang persetujuan pinjamannya lebih besar.

Secara keseluruhan, temuan ini mengindikasikan bahwa **homeownership** merupakan prediktor penting dalam persetujuan pinjaman, karena merefleksikan stabilitas finansial debitur, dan pola ini konsisten di antara kedua gender.

In [ ]:
# Copy data
plot_df = df.copy()

# Label mapping for loan status
status_labels = {
    0: '0 = Rejected',
    1: '1 = Approved',
    '0': '0 = Rejected',
    '1': '1 = Approved'
}

plot_df['loan_status_label'] = plot_df['loan_status'].map(status_labels).fillna(
    plot_df['loan_status'].astype(str)
)

# Count data by gender, loan intent, and loan status
count_df = (
    plot_df
    .groupby(['person_gender', 'loan_intent', 'loan_status_label'])
    .size()
    .reset_index(name='count')
)

# Category orders
genders = plot_df['person_gender'].dropna().unique()
loan_intent_order = plot_df['loan_intent'].dropna().unique()
loan_status_order = ['0 = Rejected', '1 = Approved']

# Colors
colors = px.colors.qualitative.Plotly[:len(loan_status_order)]
color_map = dict(zip(loan_status_order, colors))

# Create subplots by gender
fig = make_subplots(
    rows=1,
    cols=len(genders),
    subplot_titles=[f'Gender: {gender}' for gender in genders],
    horizontal_spacing=0.12
)

# Add grouped bar charts
for col_idx, gender in enumerate(genders, start=1):
    gender_df = count_df[count_df['person_gender'] == gender]

    for status in loan_status_order:
        status_df = gender_df[gender_df['loan_status_label'] == status]

        # Ensure all loan intent categories appear
        status_df = (
            pd.DataFrame({'loan_intent': loan_intent_order})
            .merge(status_df, on='loan_intent', how='left')
        )

        status_df['count'] = status_df['count'].fillna(0).astype(int)

        fig.add_trace(
            go.Bar(
                x=status_df['loan_intent'],
                y=status_df['count'],
                text=status_df['count'],
                textposition='outside',
                name=status,
                marker_color=color_map[status],
                showlegend=True if col_idx == 1 else False,
                cliponaxis=False
            ),
            row=1,
            col=col_idx
        )

    fig.update_xaxes(
        title_text='Loan Intent',
        title_standoff=20,
        categoryorder='array',
        categoryarray=loan_intent_order,
        tickangle=90,
        row=1,
        col=col_idx
    )

    fig.update_yaxes(
        title_text='Count' if col_idx == 1 else '',
        title_standoff=15,
        row=1,
        col=col_idx
    )

# Add extra y-axis space for count labels
max_count = count_df['count'].max()

for col_idx in range(1, len(genders) + 1):
    fig.update_yaxes(
        range=[0, max_count * 1.25],
        row=1,
        col=col_idx
    )

# Layout settings
fig.update_layout(
    title=dict(
        text='Loan Approval Status by Loan Intent and Gender',
        x=0.5,
        xanchor='center',
        y=0.98,
        yanchor='top'
    ),
    height=700,
    # width=1200,
    margin=dict(
        t=150,
        b=160,
        l=80,
        r=70
    ),
    barmode='group',
    bargap=0.25,
    bargroupgap=0.08,
    showlegend=True,
    legend=dict(
        title='Loan Status',
        orientation='h',
        x=0.5,
        xanchor='center',
        y=1.08,
        yanchor='bottom'
    )
)

# Adjust subplot title spacing
for annotation in fig.layout.annotations:
    annotation.update(
        y=1.02,
        yanchor='bottom',
        font=dict(size=14)
    )

fig.show()

***Insights dari catplot **loan intent vs. loan status by gender**:***

1. **Loan Intent and Approval Likelihood**:
   - Pada debitur laki‑laki maupun perempuan, tujuan pinjaman seperti debt consolidation dan personal loan menunjukkan jumlah penolakan yang lebih tinggi dibanding persetujuan, sehingga mengindikasikan bahwa tujuan tersebut dipandang memiliki risiko lebih besar.
   - Sebaliknya, tujuan pinjaman seperti education dan home improvement memiliki rasio persetujuan terhadap penolakan yang relatif lebih seimbang, sehingga cenderung dipandang lebih positif oleh pemberi pinjaman.

2. **Gender Differences in Loan Intent**:
   - Kedua gender memperlihatkan tren yang serupa untuk masing‑masing tujuan pinjaman, meskipun terdapat beberapa perbedaan. Misalnya, debitur perempuan dengan tujuan pendidikan dan keperluan medis tampak memiliki tingkat persetujuan yang relatif tinggi, sementara debitur laki‑laki terlihat lebih banyak disetujui untuk pinjaman dengan tujuan venture.
   - Hal ini dapat mencerminkan kebijakan persetujuan yang berbeda berdasarkan kombinasi tujuan pinjaman dan profil demografis.

3. **High Rejection for Debt Consolidation**:
   - Pinjaman dengan tujuan debt consolidation memiliki tingkat penolakan yang menonjol pada kedua gender. Pola ini menunjukkan bahwa pemberi pinjaman mungkin mengaitkan tujuan ini dengan risiko finansial yang lebih tinggi, misalnya akibat riwayat utang sebelumnya.

4. **Approved Applications for Productive Purposes**:
   - Tujuan pinjaman yang bersifat produktif atau membangun aset, seperti home improvement dan education, memiliki tingkat persetujuan yang relatif lebih tinggi.
   - Hal ini mengindikasikan bahwa pemberi pinjaman memandang tujuan‑tujuan tersebut sebagai bentuk investasi terhadap stabilitas dan kapasitas finansial jangka panjang debitur.

Secara keseluruhan, temuan ini menguatkan bahwa loan intent merupakan faktor penting dalam keputusan persetujuan pinjaman, di mana beberapa tujuan dipersepsikan lebih berisiko dan berpengaruh pada keputusan di kedua gender.

In [ ]:
# Copy data
plot_df = df.copy()

# Label mapping for loan status
status_labels = {
    0: '0 = Rejected',
    1: '1 = Approved',
    '0': '0 = Rejected',
    '1': '1 = Approved'
}

plot_df['loan_status_label'] = plot_df['loan_status'].map(status_labels).fillna(
    plot_df['loan_status'].astype(str)
)

# Count data by gender, previous loan defaults, and loan status
count_df = (
    plot_df
    .groupby(['person_gender', 'previous_loan_defaults_on_file', 'loan_status_label'])
    .size()
    .reset_index(name='count')
)

# Category orders
genders = plot_df['person_gender'].dropna().unique()
default_order = plot_df['previous_loan_defaults_on_file'].dropna().unique()
loan_status_order = ['0 = Rejected', '1 = Approved']

# Colors
colors = px.colors.qualitative.Plotly[:len(loan_status_order)]
color_map = dict(zip(loan_status_order, colors))

# Create subplots by gender
fig = make_subplots(
    rows=1,
    cols=len(genders),
    subplot_titles=[f'Gender: {gender}' for gender in genders],
    horizontal_spacing=0.12
)

# Add grouped bar charts
for col_idx, gender in enumerate(genders, start=1):
    gender_df = count_df[count_df['person_gender'] == gender]

    for status in loan_status_order:
        status_df = gender_df[gender_df['loan_status_label'] == status]

        # Ensure all previous loan default categories appear
        status_df = (
            pd.DataFrame({'previous_loan_defaults_on_file': default_order})
            .merge(status_df, on='previous_loan_defaults_on_file', how='left')
        )

        status_df['count'] = status_df['count'].fillna(0).astype(int)

        fig.add_trace(
            go.Bar(
                x=status_df['previous_loan_defaults_on_file'],
                y=status_df['count'],
                text=status_df['count'],
                textposition='outside',
                name=status,
                marker_color=color_map[status],
                showlegend=True if col_idx == 1 else False,
                cliponaxis=False
            ),
            row=1,
            col=col_idx
        )

    fig.update_xaxes(
        title_text='Previous Loan Defaults',
        title_standoff=15,
        categoryorder='array',
        categoryarray=default_order,
        row=1,
        col=col_idx
    )

    fig.update_yaxes(
        title_text='Count' if col_idx == 1 else '',
        title_standoff=15,
        row=1,
        col=col_idx
    )

# Add extra y-axis space for count labels
max_count = count_df['count'].max()

for col_idx in range(1, len(genders) + 1):
    fig.update_yaxes(
        range=[0, max_count * 1.25],
        row=1,
        col=col_idx
    )

# Layout settings
fig.update_layout(
    title=dict(
        text='Loan Approval Status by Previous Loan Defaults and Gender',
        x=0.5,
        xanchor='center',
        y=0.98,
        yanchor='top'
    ),
    height=600,
    margin=dict(
        t=150,
        b=90,
        l=80,
        r=70
    ),
    barmode='group',
    bargap=0.25,
    bargroupgap=0.08,
    showlegend=True,
    legend=dict(
        title='Loan Status',
        orientation='h',
        x=0.5,
        xanchor='center',
        y=1.08,
        yanchor='bottom'
    )
)

# Adjust subplot title spacing
for annotation in fig.layout.annotations:
    annotation.update(
        y=1.02,
        yanchor='bottom',
        font=dict(size=14)
    )

fig.show()

***Insights for **previous loan defaults vs. loan status by gender**:***

1. **Impact of Previous Defaults on Loan Approval**:
   - Debitur dengan riwayat gagal bayar pinjaman sebelumnya memiliki jumlah penolakan yang jauh lebih tinggi pada kedua gender, yang menunjukkan bahwa riwayat gagal bayar merupakan faktor negatif yang sangat kuat dalam keputusan persetujuan pinjaman.
   - Debitur tanpa riwayat gagal bayar terlihat memperoleh persetujuan pinjaman yang jauh lebih banyak, menegaskan bahwa rekam jejak kredit yang bersih terkait dengan tingkat persetujuan yang lebih tinggi.

2. **Gender-Specific Observations**:
   - Baik debitur laki‑laki maupun perempuan tanpa riwayat gagal bayar menunjukkan pola persetujuan dan penolakan yang serupa, sehingga pengaruh riwayat gagal bayar tampak konsisten di antara kedua gender.
   - Di antara debitur yang memiliki riwayat gagal bayar, tingkat penolakan sangat tinggi untuk kedua gender, memperkuat pentingnya rekam jejak kredit yang bersih di mata pemberi pinjaman.

3. **High Rejection for Defaults**:
   - Tingginya tingkat penolakan bagi pemohon dengan gagal bayar sebelumnya menunjukkan bahwa pemberi pinjaman memandang kelompok ini sebagai berisiko tinggi, terlepas dari faktor lain, sehingga sangat memengaruhi hasil status pinjaman.

Pola‑pola ini menegaskan bahwa **previous loan defaults** merupakan faktor krusial dalam persetujuan pinjaman, dan menjaga riwayat kredit tetap bersih sangat menguntungkan bagi pemohon laki‑laki maupun perempuan saat mengajukan pinjaman.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Loan Amount Distribution by Loan Status",
        "Loan Interest Rate Distribution by Loan Status"
    )
)

# Violin plot loan_amnt
fig.add_trace(
    go.Violin(
        x=df["loan_status"],
        y=df["loan_amnt"],
        box_visible=True,
        meanline_visible=True,
        points="all",
        name="Loan Amount"
    ),
    row=1, col=1
)

# Violin plot loan_int_rate
fig.add_trace(
    go.Violin(
        x=df["loan_status"],
        y=df["loan_int_rate"],
        box_visible=True,
        meanline_visible=True,
        points="all",
        name="Interest Rate"
    ),
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text='Loan Amount and Interest Rate Distribution by Loan Status',
        x=0.5,
        xanchor='center',
        y=0.98,
        yanchor='top'
    ),
    height=600,
    margin=dict(
        t=120,
        b=80,
        l=80,
        r=70
    ),
    showlegend=True,
    legend=dict(
        title='Variable',
        orientation='h',
        x=0.5,
        xanchor='center',
        y=1.08,
        yanchor='bottom'
    )
)

fig.update_xaxes(title_text="Loan Status", row=1, col=1)
fig.update_yaxes(title_text="Loan Amount", row=1, col=1)

fig.update_xaxes(title_text="Loan Status", row=1, col=2)
fig.update_yaxes(title_text="Loan Interest Rate", row=1, col=2)

fig.show()

Insights untuk **loan amount** and **loan interest rate** berdasarkan loan status:

1. **Loan Amount Distribution by Loan Status**:
   - Distribusi jumlah pinjaman berbeda antara pinjaman yang disetujui dan yang ditolak.
   - Pinjaman yang disetujui menunjukkan sebaran yang lebih luas pada berbagai besaran pinjaman, dengan kecenderungan pusat di kisaran nilai menengah.
   - Pinjaman yang ditolak cenderung terkonsentrasi pada jumlah pinjaman yang sangat kecil maupun sangat besar, yang mengindikasikan bahwa permintaan pinjaman terlalu kecil atau terlalu besar lebih berisiko ditolak.

2. **Loan Interest Rate Distribution by Loan Status**:
   - Suku bunga pinjaman cenderung lebih tinggi pada pinjaman yang ditolak, dengan kecenderungan pusat di tingkat bunga yang lebih tinggi dibanding pinjaman yang disetujui.
   - Pinjaman yang disetujui memiliki distribusi suku bunga yang lebih sempit dengan rata‑rata yang lebih rendah, yang menyiratkan bahwa pinjaman dengan risiko lebih rendah (suku bunga lebih rendah) memiliki peluang persetujuan lebih besar.

In [ ]:
# Define numerical columns with target
numerical_columns_with_target = [
    'person_age',
    'person_income',
    'person_emp_exp',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length',
    'credit_score'
]

# Create pairplot of numerical features with loan_status as hue
sns.pairplot(df[numerical_columns_with_target + ['loan_status']],
             hue='loan_status',
             palette='muted'
            )
plt.show()

# 🔢 **Data Preprocessing**

In [ ]:
# Binary Encoding for person_gender
df['person_gender'] = df['person_gender'].map({'female': 0, 'male': 1})

# Binary Encoding for previous_loan_defaults_on_file
df['previous_loan_defaults_on_file'] = df['previous_loan_defaults_on_file'].map({'No': 0, 'Yes': 1})

# Ordinal Encoding for person_education (if applicable)
education_order = {'High School': 1, 'Associate': 2, 'Bachelor': 3, 'Master': 4, 'Doctorate': 5}
df['person_education'] = df['person_education'].map(education_order)

# One-Hot Encoding for person_home_ownership and loan_intent
df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent'], drop_first=True)

# Display the transformed DataFrame
print(df.head())

The transformation applied to the dataset:

1. **Binary Encoding**:
   - `person_gender` sekarang direpresentasikan sebagai `0` (female) dan `1` (male).
   - `previous_loan_defaults_on_file` sekarang direpresentasikan sebagai `0` (No) dan `1` (Yes).

2. **Ordinal Encoding**:
   - `person_education` dipetakan berdasarkan jenjang pendidikan, dengan nilai yang lebih tinggi menunjukkan tingkat pendidikan yang lebih tinggi  (e.g., `High School` = 1, `Doctorate` = 5).

3. **One-Hot Encoding**:
   - Kolom baru dibuat untuk `person_home_ownership` dan `loan_intent`, masing‑masing merepresentasikan kategori unik (e.g., `person_home_ownership_OWN`, `loan_intent_PERSONAL`), dengan satu kategori dihapus untuk menghindari multikolinearitas dalam transformasi.

In [ ]:
# Replacing Outliers with Median
median_age = df['person_age'].median()
df['person_age'] = df['person_age'].apply(lambda x: median_age if x > 100 else x)

- Nilai usia maksimum 144 jelas merupakan outlier karena melampaui rentang usia manusia yang wajar. Untuk menanganinya, dilakukan:

    - **Replacing Outliers with Median:** Ganti nilai usia di atas ambang tertentu (e.g., 100) dengan nilai median usia pada dataset, sehingga distribusi usia tetap realistis dan tidak terlalu terdistorsi oleh nilai ekstrem.

In [ ]:
# Analyze the 'person_age' column
# Display summary statistics for person_age
print(f'\nSummary Statistics for person_age:\n\n', df['person_age'].describe())

Setelah mengganti outlier dengan median usia:

   - Nilai usia maksimum sekarang menjadi **94**, yang masih berada dalam rentang wajar untuk data usia manusia.
   - Rata‑rata usia (27.75) dan standar deviasi (5.91) sedikit menurun, menunjukkan distribusi usia yang lebih rapat.
   - Penggantian nilai ekstrem dengan median membantu menghilangkan nilai yang tidak realistis tanpa menghapus baris data, sehingga integritas dataset tetap terjaga.
   - Pendekatan ini mempertahankan sebaran usia `person_age` yang realistis, dengan median yang tetap di angka 26.

**Correlation Heatmap**

In [ ]:
# matrix correlation
corr_matrix = df.corr()

# Plotting the heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Extract correlation values with respect to the target variable (loan_status)
target_variable = 'loan_status'
target_corr = corr_matrix[[target_variable]].sort_values(by=target_variable, ascending=False)

# Plotting the heatmap for correlation values with respect to the target variable
plt.figure(figsize=(4, 6))
sns.heatmap(target_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title(f'Correlation with {target_variable}')
plt.show()

Nilai korelasi setiap fitur terhadap variabel target `loan_status` memberikan gambaran fitur mana yang berpotensi menjadi prediktor penting untuk persetujuan pinjaman.

***Positively Correlated Features with `loan_status`***

1. **`loan_percent_income` (0.38)**: Ini adalah korelasi positif terkuat dengan `loan_status`, yang mengindikasikan bahwa debitur dengan rasio jumlah pinjaman terhadap pendapatan yang lebih tinggi justru cenderung memiliki peluang persetujuan yang lebih besar.

2. **`loan_int_rate` (0.33)**: Suku bunga yang lebih tinggi berkorelasi positif dengan status persetujuan, yang bisa menunjukkan bahwa debitur dengan profil yang lebih berisiko (dengan suku bunga lebih tinggi) tetap cukup sering disetujui.

3. **`person_home_ownership_RENT` (0.26)**: Status menyewa memiliki korelasi positif, yang mengisyaratkan bahwa penyewa mungkin memiliki tingkat persetujuan sedikit lebih tinggi dibanding status kepemilikan rumah lainnya.


***Weak Positive Correlations***

- **`loan_amnt` (0.11)**: Jumlah pinjaman memiliki korelasi positif lemah, menunjukkan kecenderungan kecil bahwa pinjaman dengan jumlah lebih besar lebih sering disetujui.

- **Loan intents like `MEDICAL` (0.07)** dan `HOMEIMPROVEMENT` (0.03)** juga menunjukkan korelasi positif lemah, yang menyiratkan bahwa beberapa tujuan pinjaman tertentu mungkin sedikit memengaruhi peluang persetujuan.

***Negatively Correlated Features with `loan_status`***

1. **`previous_loan_defaults_on_file` (-0.54)**: Ini merupakan korelasi negatif yang paling kuat, menandakan bahwa riwayat gagal bayar pinjaman sebelumnya menjadi faktor signifikan yang menurunkan peluang persetujuan.

2. **`person_income` (-0.14)**: Orang dengan pendapatan yang lebih tinggi justru berkorelasi negatif lemah dengan persetujuan, yang tampak agak berlawanan dengan intuisi. Hal ini bisa berarti debitur berpendapatan lebih tinggi cenderung mengajukan pinjaman yang lebih berisiko, atau debitur berpendapatan lebih rendah memperoleh lebih banyak persetujuan berdasarkan kriteria tertentu.

3. **`person_home_ownership_OWN` (-0.09)**: Status rumah milik sendiri memiliki korelasi negatif kecil, yang mengindikasikan bahwa kepemilikan rumah tidak selalu lebih menguntungkan dibanding menyewa atau status lainnya dalam hal peluang persetujuan.

4. **Loan intents like `VENTURE` (-0.09)** dan **`EDUCATION` (-0.06)** berkorelasi negatif dengan persetujuan, yang mungkin mencerminkan persepsi risiko yang lebih tinggi terhadap tujuan‑tujuan tersebut.

***Implications***

- Prediktor terkuat untuk persetujuan pinjaman adalah **loan percent income**, **loan interest rate**, dan **previous loan defaults**.

- Credit score secara mengejutkan memiliki korelasi yang nyaris nol (-0,007), yang mengindikasikan bahwa skor kredit mungkin tidak berperan besar dalam keputusan persetujuan pada dataset ini, atau justru berinteraksi dengan fitur lain dengan cara yang lebih kompleks yang perlu digali lebih lanjut dalam pemodelan lanjutan.

# 📈 **Model Training and Evaluation**

In [ ]:
# Separate features and target from the train dataset
X = df.drop(['loan_status'], axis=1)
y = df['loan_status']

In [ ]:
# Split the data into training and validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Data scaling
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
# Dictionary to store the models and their names
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(verbosity=-1, random_state=42),
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    # Predictions on validation set
    y_val_pred = model.predict(X_val)

    # Train and Test Scores
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_val, y_val)

    # Accuracy Score
    accuracy = accuracy_score(y_val, y_val_pred)

    results.append({
        'Model': name,
        'Train Score': train_score,
        'Test Score': test_score,
        'Accuracy Score': accuracy
    })

    # Classification report
    print(f"Classification Report for {name}:\n")
    print(classification_report(y_val, y_val_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_val, y_val_pred)

    # Plotting confusion matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='PuBu', xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
    plt.title(f'Confusion Matrix for {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    print("\n" + "="*60 + "\n")

results_df = pd.DataFrame(results)

# Display the model performance table
print("Model Performance Table:")
display(results_df)

***Key Observations***:

* Logistic Regression: Memiliki kinerja sedang, kemungkinan karena sifatnya yang linear sehingga tidak sepenuhnya mampu menangkap hubungan yang kompleks di dalam data.

* Random Forest, XGBoost dan LightGBM: Model‑model ini menunjukkan performa yang kuat, dengan akurasi tinggi (sekitar 0.93). Model ensemble seperti ini umumnya bekerja sangat baik pada data terstruktur karena mampu menangkap hubungan non‑linear.

* Model Terbaik: XGBoost mencapai akurasi tertinggi sebesar 0.934, sehingga menjadi pilihan utama untuk tugas ini.

In [ ]:
# Identify the best model by accuracy
best_model_row = results_df.loc[results_df['Accuracy Score'].idxmax()]
best_model_name = best_model_row['Model']
best_model_accuracy = best_model_row['Accuracy Score']

print(f"\nBest Model: {best_model_name} with Accuracy: {best_model_accuracy:.4f}")

In [ ]:
# Check if the best model supports feature importances
best_model = models[best_model_name]

if hasattr(best_model, 'feature_importances_'):
    feature_importances = best_model.feature_importances_

    feature_importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': feature_importances
    }).sort_values(by='Importance', ascending=True)

    fig = px.bar(
        feature_importance_df,
        x='Importance',
        y='Feature',
        orientation='h',
        title=f'Feature Importances - {best_model_name}',
        labels={
            'Importance': 'Importance Score',
            'Feature': 'Feature'
        }
    )

    fig.update_layout(
        title=dict(
            text=f'Feature Importances - {best_model_name}',
            x=0.5,
            xanchor='center'
        ),
        height=600,
        margin=dict(
            t=80,
            b=60,
            l=120,
            r=50
        ),
        showlegend=False
    )

    fig.show()

else:
    print(f"The {best_model_name} model does not support feature importances.")

***Feature Importance Analysis***

* Untuk XGBoost (model terbaik), ekstraksi feature importance dilakukan untuk melihat fitur mana yang paling berpengaruh dalam memprediksi persetujuan pinjaman.

* Fitur Utama: Fitur yang kemungkinan memiliki skor kepentingan paling tinggi antara lain `loan_percent_income` dan `previous_loan_defaults_on_file`, karena keduanya sangat terkait dengan kelayakan kredit pemohon dan tingkat risiko pinjaman.

In [ ]:
# Obtain predicted probabilities for the validation set
test_probabilities = best_model.predict_proba(X_val)[:, 1]

# KDE calculation
x_grid = np.linspace(0, 1, 500)
kde = gaussian_kde(test_probabilities)

# Scale KDE agar mengikuti skala Frequency, bukan Density
counts, bin_edges = np.histogram(test_probabilities, bins=30, range=(0, 1))
bin_width = bin_edges[1] - bin_edges[0]
kde_y = kde(x_grid) * len(test_probabilities) * bin_width

# Plotly figure
fig = go.Figure()

# Histogram
fig.add_trace(
    go.Histogram(
        x=test_probabilities,
        nbinsx=30,
        name='Frequency',
        opacity=0.75
    )
)

# KDE line
fig.add_trace(
    go.Scatter(
        x=x_grid,
        y=kde_y,
        mode='lines',
        name='KDE'
    )
)

fig.update_layout(
    title=dict(
        text=f'Distribution of Predicted Loan Approval Probabilities - {best_model_name}',
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title='Predicted Probability of Loan Approval',
        range=[0, 1],
        showgrid=True
    ),
    yaxis=dict(
        title='Frequency',
        showgrid=True
    ),
    height=600,
    margin=dict(
        t=90,
        b=70,
        l=80,
        r=50
    ),
    bargap=0.05,
    showlegend=True
)

fig.show()

In [ ]:
# Convert probabilities to binary predictions
binary_predictions = (test_probabilities > 0.5).astype(int)

# Create DataFrame for plotting
pred_df = pd.DataFrame({
    'Loan Status': binary_predictions.flatten()
})

# Rename labels
pred_df['Loan Status Label'] = pred_df['Loan Status'].map({
    0: 'Not Approved',
    1: 'Approved'
})

# Count distribution
count_df = pred_df['Loan Status Label'].value_counts().reindex(
    ['Not Approved', 'Approved']
).reset_index()

count_df.columns = ['Loan Status', 'Count']

# Plot with Plotly
fig = px.bar(
    count_df,
    x='Loan Status',
    y='Count',
    color='Loan Status',
    text='Count',
    title='Distribution of Predicted Loan Status',
    labels={
        'Loan Status': 'Loan Status (0: Not Approved, 1: Approved)',
        'Count': 'Count'
    }
)

fig.update_layout(
    title=dict(
        text='Distribution of Predicted Loan Status',
        x=0.5,
        xanchor='center'
    ),
    height=500,
    margin=dict(
        t=80,
        b=70,
        l=70,
        r=50
    ),
    xaxis=dict(
        categoryorder='array',
        categoryarray=['Not Approved', 'Approved']
    ),
    yaxis=dict(
        showgrid=True
    ),
    showlegend=False
)

fig.update_traces(
    textposition='outside'
)

fig.show()